In [1]:
# ============================================================
# IDENTIFY NEXT BATCH — LOCAL EXECUTION
# ============================================================

import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

# Load environment
def run_nb(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    src, _ = PythonExporter().from_notebook_node(nb)
    exec(src, globals())

run_nb("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")

control_file = f"{CONTROL_PATH}/batch_control/data.parquet"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/10 07:59:00 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/10 07:59:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/10 07:59:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [2]:
# -----------------------------------------
# 1. Landing batches (folders)
# -----------------------------------------
landing_batches = sorted([
    name for name in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, name))
])

In [3]:
# -----------------------------------------
# 2. Tracked batches
# -----------------------------------------
if os.path.exists(control_file):
    tracked_batches = [
        row.batch_id
        for row in spark.read.parquet(control_file)
            .filter(F.col("status").isin("in_progress", "completed"))
            .select("batch_id").distinct().collect()
    ]
else:
    tracked_batches = []

In [4]:
# -----------------------------------------
# 3. Identify earliest unprocessed batch
# -----------------------------------------
new_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None

print("Landing batches :", landing_batches)
print("Tracked batches :", tracked_batches)
print("Next batch      :", next_batch)

Landing batches : []
Tracked batches : []
Next batch      : None


In [5]:
# -----------------------------------------
# 4. Export values for next notebook
# -----------------------------------------
has_batch = next_batch is not None